In [1]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
FOLDER_AUDIO_POTONGAN = "../Data/segmented_audio"
FOLDER_HASIL_FITUR = "../Data/features"
os.makedirs(FOLDER_HASIL_FITUR, exist_ok=True)

In [3]:
daftar_genre = ['rock', 'jazz', 'classical', 'blues', 'pop', 'country']
kumpulan_baris_data = []

In [4]:
# Fungsi pembantu untuk mengambil nilai Mean, STD, Min, Max (Aturan Tahap 5 Dosen)
def ambil_statistik_fitur(vektor_fitur):
    return [vektor_fitur.mean(), vektor_fitur.std(), vektor_fitur.min(), vektor_fitur.max()]

print("--- MEMULAI TAHAP 4 & 5: EKSTRAKSI & FEATURE ENGINEERING LENGKAP (6 GENRE) ---")
for genre in daftar_genre:
    path_genre = os.path.join(FOLDER_AUDIO_POTONGAN, genre)
    if not os.path.isdir(path_genre): continue
        
    print(f"Mengekstrak statistik fitur untuk genre: {genre}")
    daftar_file = [f for f in os.listdir(path_genre) if f.endswith('.wav')]
    
    for nama_file in tqdm(daftar_file):
        path_file = os.path.join(path_genre, nama_file)
        try:
            y, sr = librosa.load(path_file, sr=None)
            
            # --- TAHAP 4: Ekstraksi Nilai Domain Sinyal ---
            rms = librosa.feature.rms(y=y)
            zcr = librosa.feature.zero_crossing_rate(y=y)
            centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
            bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
            rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr) # Tambahan wajib dosen
            chroma = librosa.feature.chroma_stft(y=y, sr=sr)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            
            # --- TAHAP 5: Feature Engineering Statistik (Mean, STD, Min, Max) ---
            fitur_baris = [nama_file]
            fitur_baris.extend(ambil_statistik_fitur(rms))
            fitur_baris.extend(ambil_statistik_fitur(zcr))
            fitur_baris.extend(ambil_statistik_fitur(centroid))
            fitur_baris.extend(ambil_statistik_fitur(bandwidth))
            fitur_baris.extend(ambil_statistik_fitur(rolloff))
            
            # Statistik untuk 12 Kolom Chroma
            for c in chroma:
                fitur_baris.extend(ambil_statistik_fitur(c))
                
            # Statistik untuk 13 Kolom MFCC
            for m in mfcc:
                fitur_baris.extend(ambil_statistik_fitur(m))
                
            fitur_baris.append(genre) # Label Target
            kumpulan_baris_data.append(fitur_baris)
        except:
            pass

--- MEMULAI TAHAP 4 & 5: EKSTRAKSI & FEATURE ENGINEERING LENGKAP (6 GENRE) ---
Mengekstrak statistik fitur untuk genre: rock


100%|██████████| 999/999 [00:37<00:00, 26.43it/s]


Mengekstrak statistik fitur untuk genre: jazz


100%|██████████| 990/990 [00:32<00:00, 30.16it/s]


Mengekstrak statistik fitur untuk genre: classical


...

100%|██████████| 1000/1000 [00:34<00:00, 29.32it/s]


Mengekstrak statistik fitur untuk genre: pop


100%|██████████| 1000/1000 [00:37<00:00, 26.91it/s]


Mengekstrak statistik fitur untuk genre: country


100%|██████████| 997/997 [00:37<00:00, 26.80it/s]


In [5]:
# Menyusun nama header kolom secara otomatis
nama_kolom = ['nama_file']
for f in ['rms', 'zcr', 'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff']:
    nama_kolom.extend([f'{f}_mean', f'{f}_std', f'{f}_min', f'{f}_max'])
for i in range(12):
    nama_kolom.extend([f'chroma_{i}_mean', f'chroma_{i}_std', f'chroma_{i}_min', f'chroma_{i}_max'])
for i in range(13):
    nama_kolom.extend([f'mfcc_{i}_mean', f'mfcc_{i}_std', f'mfcc_{i}_min', f'mfcc_{i}_max'])
nama_kolom.append('label_genre')

tabel_final = pd.DataFrame(kumpulan_baris_data, columns=nama_kolom)
tabel_final.to_csv(os.path.join(FOLDER_HASIL_FITUR, "combined_features.csv"), index=False)
print(f"\n[SUKSES] Ekstraksi selesai!")


[SUKSES] Ekstraksi selesai!
